# Random Forest
dataset: Body Performance Data (kaggle)

predicting the fitness class (A, B, C, D) of a person using random forest. A is the best and D is the worst

i used the same data as the logistic regression assignment but this time with all 4 classes not binary

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

## Load the data

In [ ]:
df = pd.read_csv('/kaggle/input/body-performance-data/bodyPerformance.csv')
df.head()

In [ ]:
print(df.shape)
print(df.isnull().sum().sum())   # no missing values
print(df.duplicated().sum())     # 1 duplicate

In [ ]:
df['class'].value_counts()

the 4 classes are almost the same size so its balanced

In [ ]:
df.describe()

some values look wrong, like diastolic and systolic = 0 and gripForce = 0, and the max of sit and bend forward is 213 cm which is impossible. so i will remove these rows

## Preprocessing
- remove the duplicate
- remove the wrong values
- change gender to numbers (M = 1, F = 0)

i didnt use scaling because random forest doesnt need it

In [ ]:
df = df.drop_duplicates()

df = df[df['diastolic'] >= 40]
df = df[df['systolic'] >= 70]
df = df[df['gripForce'] > 0]
df = df[df['broad jump_cm'] > 0]
df = df[df['sit and bend forward_cm'] <= 60]

df['gender'] = df['gender'].map({'M': 1, 'F': 0})

print(df.shape)

In [ ]:
X = df.drop('class', axis=1)
y = df['class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape)

## Random Forest (default)
first i tried it with the default values

In [ ]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print('train accuracy:', accuracy_score(y_train, model.predict(X_train)))
print('test accuracy :', accuracy_score(y_test, y_pred))

train accuracy is 1 and test is lower so its overfitting. i think its because max_depth is None by default so the trees keep growing

## Gini vs Entropy

In [ ]:
for c in ['gini', 'entropy']:
    start = time.time()
    m = RandomForestClassifier(criterion=c, random_state=42)
    m.fit(X_train, y_train)
    end = time.time()
    print(c, '-> accuracy:', round(accuracy_score(y_test, m.predict(X_test)), 4), ' time:', round(end - start, 2), 's')

almost the same accuracy, entropy was a little better but gini is faster

## Trying different max_depth

In [ ]:
depths = [2, 4, 6, 8, 10, 12, 15, 20]
train_acc = []
test_acc = []

for d in depths:
    m = RandomForestClassifier(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, m.predict(X_train)))
    test_acc.append(accuracy_score(y_test, m.predict(X_test)))

plt.plot(depths, train_acc, marker='o', label='train')
plt.plot(depths, test_acc, marker='o', label='test')
plt.xlabel('max_depth')
plt.ylabel('accuracy')
plt.legend()
plt.title('max_depth vs accuracy')
plt.show()

after depth 12 the test accuracy doesnt really go up anymore but train keeps going up, so deeper trees just overfit

## Hyperparameter tuning
using GridSearchCV to find the best values

In [ ]:
params = {
    'n_estimators': [100, 200],
    'max_depth': [10, 15, 20],
    'max_features': ['sqrt', 'log2'],
    'min_samples_split': [2, 5]
}

grid = GridSearchCV(RandomForestClassifier(random_state=42), params, cv=3)
grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)

## Final model

In [ ]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print('Accuracy :', round(accuracy_score(y_test, y_pred), 4))
print('Precision:', round(precision_score(y_test, y_pred, average='macro'), 4))
print('Recall   :', round(recall_score(y_test, y_pred, average='macro'), 4))
print('F1       :', round(f1_score(y_test, y_pred, average='macro'), 4))

i used average='macro' because there are 4 classes

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=best_model.classes_).plot()
plt.title('confusion matrix')
plt.show()

class A and D are the easiest. most of the mistakes are with the class next to it, like B gets predicted as A or C. it almost never mixes A and D

## Feature importance

In [ ]:
importance = pd.Series(best_model.feature_importances_, index=X.columns)
importance.sort_values().plot(kind='barh')
plt.title('feature importance')
plt.show()

sit and bend forward is the most important feature then sit ups. gender is the least important

## Conclusion
- test accuracy with default random forest was 72.4%
- after tuning with GridSearchCV it went up to 73.4%
- the model overfits when the trees are too deep
- gini and entropy gave almost the same result
- B and C are the hardest classes to predict because they are in the middle